In [1]:
import pandas as pd
import os
import sys
import subprocess

In [2]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'plotly', 'pandas', 'matplotlib', 'streamlit', 'openpyxl'])

CompletedProcess(args=['f:\\PROJECTS\\PERSONAL_FINANCE_ADVISER\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'plotly', 'pandas', 'matplotlib', 'streamlit', 'openpyxl'], returncode=0)

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

spending_df = pd.read_csv('../data/spending_clean.csv', parse_dates=['Date'])
budget_df = pd.read_csv('../data/Budget.csv')

print("✅ Data loaded")
print(f"Spending rows: {len(spending_df)}")
print(f"Budget rows: {len(budget_df)}")
print(spending_df.head(3))

✅ Data loaded
Spending rows: 617
Budget rows: 19
        Date       Description   Amount Transaction Type         Category  \
0 2018-01-01            Amazon    11.11            debit         Shopping   
1 2018-01-02  Mortgage Payment  1247.44            debit  Mortgage & Rent   
2 2018-01-02   Thai Restaurant    24.22            debit      Restaurants   

    Account Name  Month Month_Name  Year  
0  Platinum Card      1    January  2018  
1       Checking      1    January  2018  
2    Silver Card      1    January  2018  


In [4]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'nbformat', 'ipython'])

CompletedProcess(args=['f:\\PROJECTS\\PERSONAL_FINANCE_ADVISER\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'nbformat', 'ipython'], returncode=0)

In [5]:
# ============================================
# CHART 1 — Monthly Spending Trend
# ============================================

monthly = spending_df.groupby(['Month', 'Month_Name'])['Amount'].sum().reset_index()
monthly = monthly.sort_values('Month')

fig = px.line(
    monthly,
    x='Month_Name',
    y='Amount',
    title='Monthly Spending Trend — 2018',
    markers=True,
    labels={'Month_Name': 'Month', 'Amount': 'Total Spent ($)'}
)

fig.update_traces(
    line=dict(color='#6366f1', width=3),
    marker=dict(size=8)
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis_tickprefix='$',
    yaxis_tickformat=',.0f',
    hovermode='x unified'
)

fig.show()

In [6]:
by_category = spending_df.groupby('Category')['Amount'].sum().reset_index()
by_category = by_category.sort_values('Amount', ascending=True)

fig = px.bar(
    by_category,
    x='Amount',
    y='Category',
    orientation='h',
    title='Total Spending by Category — 2018',
    labels={'Amount': 'Total Spent ($)', 'Category': ''},
    color='Amount',
    color_continuous_scale=[
        [0, '#c7d2fe'],
        [0.5, '#6366f1'],
        [1, '#3730a3']
    ]
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis_tickprefix='$',
    xaxis_tickformat=',.0f',
    coloraxis_showscale=False,
    height=500
)

fig.show()

In [7]:
# ============================================
# CHART 2 — Spending by Category
# ============================================

by_category = spending_df.groupby('Category')['Amount'].sum().reset_index()
by_category = by_category.sort_values('Amount', ascending=True)

fig = px.bar(
    by_category,
    x='Amount',
    y='Category',
    orientation='h',
    title='Total Spending by Category — 2018',
    labels={'Amount': 'Total Spent ($)', 'Category': ''},
    color='Amount',
    color_continuous_scale=[
        [0, '#c7d2fe'],
        [0.5, '#6366f1'],
        [1, '#3730a3']
    ]
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis_tickprefix='$',
    xaxis_tickformat=',.0f',
    coloraxis_showscale=False,
    height=500
)

fig.show()

In [8]:
# ============================================
# CHART 3 — Actual vs Budget (Fixed)
# ============================================

actual = spending_df.groupby('Category')['Amount'].sum().reset_index()
actual.columns = ['Category', 'Actual']

comparison = actual.merge(budget_df, on='Category', how='left')
comparison = comparison.dropna(subset=['Budget'])

# Fix: multiply monthly budget × 12 to get annual budget
comparison['Annual_Budget'] = comparison['Budget'] * 12
comparison = comparison.sort_values('Actual', ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Actual Spent',
    x=comparison['Category'],
    y=comparison['Actual'],
    marker_color='#6366f1'
))

fig.add_trace(go.Bar(
    name='Annual Budget',
    x=comparison['Category'],
    y=comparison['Annual_Budget'],
    marker_color='#f97316'  # orange — clearly visible
))

fig.update_layout(
    title='Actual Spending vs Annual Budget by Category — 2018',
    barmode='group',
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis_tickprefix='$',
    yaxis_tickformat=',.0f',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)

fig.show()

In [9]:
# ============================================
# CHART 4 — Anomaly Detection (Final Version)
# ============================================

# Recalculate everything fresh
category_avg = spending_df.groupby('Category')['Amount'].mean()
spending_df['Category_Avg'] = spending_df['Category'].map(category_avg)
spending_df['Is_Anomaly'] = spending_df['Amount'] > (spending_df['Category_Avg'] * 2)
spending_df['Times_Over'] = (spending_df['Amount'] / spending_df['Category_Avg']).round(1)

anomalies = spending_df[spending_df['Is_Anomaly']].copy()
anomalies = anomalies.sort_values('Times_Over', ascending=True).tail(12)
anomalies['Label'] = anomalies['Description'] + ' (' + anomalies['Category'] + ')'

fig = px.bar(
    anomalies,
    x='Times_Over',
    y='Label',
    orientation='h',
    title='Top Anomalies — How Many Times Over Category Average',
    labels={'Times_Over': 'Times Over Average', 'Label': ''},
    color='Times_Over',
    color_continuous_scale=[
        [0, '#fca5a5'],
        [0.5, '#ef4444'],
        [1, '#991b1b']
    ],
    text='Times_Over'
)

fig.update_traces(
    texttemplate='%{text}x',
    textposition='outside'
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    coloraxis_showscale=False,
    height=500,
    margin=dict(l=220, r=80),
    xaxis_title='Times Over Category Average',
    xaxis=dict(ticksuffix='x')
)

fig.show()